# Face Attendance — Multi-dataset Data Pipeline

Mỗi dataset được xử lý trong một cell riêng và ghi manifest độc lập. Không gộp nhãn identity giữa các dataset.

```text
Kaggle inputs
   ├─ Masked faces ───────> recognition_masked.csv
   ├─ WebFace 112×112 ─────> recognition_webface.csv
   ├─ WIDER YOLO ─────────> detection_wider.csv
   └─ Livesess liveness ───> liveness.csv
                                  │
                                  ├─ quality checks / deterministic split
                                  ├─ optional face alignment
                                  └─ train detection, recognition, liveness separately
```


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOTS = {
    'masked': Path('/kaggle/input/datasets/nanimasaka/masked-face-recognition-dataset'),
    'webface': Path('/kaggle/input/datasets/yakhyokhuja/webface-112x112'),
    'wider': Path('/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training'),
    'liveness': Path('/kaggle/input/datasets/harshalakshan/liveness-detection-dataset'),
}
WORK_DIR = Path('/kaggle/working/attendance-data')
MANIFEST_DIR = WORK_DIR / 'manifests'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SEED = 42
RUN_ALIGNMENT = False  # Enable only after inspecting the manifests.

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print('Inputs:', [str(path) for path in INPUT_ROOT.iterdir()])


In [ ]:
def locate_dataset(slug: str) -> Path:
    matches = [path for path in INPUT_ROOT.rglob(slug) if path.is_dir()]
    if not matches:
        raise FileNotFoundError(f'Dataset {slug!r} is not attached below {INPUT_ROOT}')
    return min(matches, key=lambda path: len(path.parts))


def find_dir(root: Path, name: str) -> Path:
    if root.name == name:
        return root
    match = next((path for path in root.rglob(name) if path.is_dir()), None)
    if match is None:
        raise FileNotFoundError(f'Could not find {name!r} below {root}')
    return match


def image_size(path: Path) -> tuple[int, int] | None:
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            return image.size
    except Exception:
        return None


def split_for(key: str) -> str:
    value = int(hashlib.sha256(f'{SEED}:{key}'.encode()).hexdigest()[:8], 16) % 100
    return 'train' if value < 70 else 'val' if value < 85 else 'test'


def write_manifest(name: str, rows: list[dict]) -> Path:
    path = MANIFEST_DIR / name
    fields = sorted({field for row in rows for field in row})
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f'{path.name}: {len(rows):,} valid rows')
    return path


def show_counts(title: str, rows: list[dict], field: str) -> None:
    counts = {}
    for row in rows:
        counts[row[field]] = counts.get(row[field], 0) + 1
    plt.figure(figsize=(6, 3))
    plt.bar(counts.keys(), counts.values())
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# 1. Masked Face Recognition: p1/, p2/, ... are person identities.
masked_dataset = DATASET_ROOTS['masked']
masked_root = find_dir(masked_dataset, 'p1').parent
masked_rows = []
for identity_dir in sorted(path for path in masked_root.iterdir() if path.is_dir() and path.name.startswith('p')):
    for image_path in sorted(identity_dir.rglob('*')):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        size = image_size(image_path)
        if size is None:
            continue
        masked_rows.append({
            'path': str(image_path), 'identity': f'masked::{identity_dir.name}',
            'split': split_for(str(image_path)), 'width': size[0], 'height': size[1],
            'source': 'masked-face-recognition-dataset', 'task': 'recognition', 'mask': 1,
        })
write_manifest('recognition_masked.csv', masked_rows)
show_counts('Masked recognition split', masked_rows, 'split')


In [ ]:
# 2. WebFace: aligned 112×112 identities for ArcFace embedding training.
MAX_RECOGNITION_IDENTITIES = 1500
MAX_IMAGES_PER_IDENTITY = 25
webface_dataset = DATASET_ROOTS['webface']
webface_root = find_dir(webface_dataset, 'webface_112x112')
webface_rows = []
for identity_dir in sorted(path for path in webface_root.iterdir() if path.is_dir())[:MAX_RECOGNITION_IDENTITIES]:
    image_paths = [path for path in sorted(identity_dir.iterdir()) if path.suffix.lower() in IMAGE_EXTENSIONS][:MAX_IMAGES_PER_IDENTITY]
    for image_path in image_paths:
        size = image_size(image_path)
        if size is None:
            continue
        webface_rows.append({
            'path': str(image_path), 'identity': f'webface::{identity_dir.name}',
            'split': split_for(str(image_path)), 'width': size[0], 'height': size[1],
            'source': 'webface-112x112', 'task': 'recognition', 'mask': 0,
        })
write_manifest('recognition_webface.csv', webface_rows)
show_counts('WebFace recognition split', webface_rows, 'split')


In [ ]:
# 3. WIDER Face Detection: preserve YOLO labels; do not use it as an identity dataset.
wider_dataset = DATASET_ROOTS['wider']
image_root = find_dir(wider_dataset, 'images')
label_root = find_dir(wider_dataset, 'labels')
detection_rows = []
for image_path in sorted(image_root.rglob('*')):
    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    relative = image_path.relative_to(image_root).with_suffix('.txt')
    label_path = label_root / relative
    if not label_path.exists():
        continue
    size = image_size(image_path)
    if size is None:
        continue
    labels = [line for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    valid = [line for line in labels if len(line.split()) == 5]
    if not valid:
        continue
    detection_rows.append({
        'image_path': str(image_path), 'label_path': str(label_path), 'faces': len(valid),
        'split': split_for(str(image_path)), 'width': size[0], 'height': size[1],
        'source': 'wider-face-for-yolo-training', 'task': 'detection', 'label_format': 'yolo',
    })
write_manifest('detection_wider.csv', detection_rows)
show_counts('WIDER detection split', detection_rows, 'split')


In [ ]:
# 4. Livesess liveness: use explicit real/fake directories; preserve native test split.
liveness_dataset = DATASET_ROOTS['liveness']
liveness_root = find_dir(liveness_dataset, 'livesess data')

def liveness_label(path: Path) -> int | None:
    parts = {part.lower() for part in path.parts}
    if parts & {'real', 'live', 'genuine'}:
        return 1
    if parts & {'fake', 'spoof', 'attack', 'print', 'replay'}:
        return 0
    return None

def liveness_split(path: Path) -> str:
    parts = {part.lower() for part in path.parts}
    if 'test' in parts:
        return 'test'
    return split_for(str(path))

liveness_rows = []
for image_path in sorted(liveness_root.rglob('*')):
    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    label = liveness_label(image_path)
    size = image_size(image_path)
    if label is None or size is None:
        continue
    liveness_rows.append({
        'path': str(image_path), 'label': label, 'origin': 'image',
        'split': liveness_split(image_path), 'width': size[0], 'height': size[1],
        'source': 'liveness-detection-dataset', 'task': 'liveness',
    })
write_manifest('liveness.csv', liveness_rows)
show_counts('Livesess labels (1=real, 0=fake)', liveness_rows, 'label')


In [ ]:
# 5. Optional alignment. Run only after the detector is selected and manifests are audited.
if RUN_ALIGNMENT:
    import subprocess
    subprocess.run(['pip', '-q', 'install', 'insightface', 'onnxruntime'], check=True)
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    print('Detector loaded. Add a crop-export loop here only for approved biometric data.')

summary = {
    'masked_recognition_rows': len(masked_rows),
    'webface_recognition_rows': len(webface_rows),
    'wider_detection_rows': len(detection_rows),
    'liveness_rows': len(liveness_rows),
    'manifest_dir': str(MANIFEST_DIR),
}
(WORK_DIR / 'preprocessing_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


## Training cells

Bật từng cờ `RUN_*` bên dưới. Recognition, detection và liveness là ba mô hình độc lập.

In [ ]:
# Training profiles. Do not use detection resolution for face embeddings.
RUN_RECOGNITION_TRAINING = True
RUN_DETECTION_TRAINING = True
RUN_LIVENESS_TRAINING = True

RECOGNITION_IMAGE_SIZE, RECOGNITION_BATCH_SIZE = 112, 160
MIN_IMAGES_PER_IDENTITY = 5
DETECTION_IMAGE_SIZE, DETECTION_BATCH_SIZE = 640, 16  # 160 at 640 usually exceeds 16 GB VRAM.
LIVENESS_IMAGE_SIZE, LIVENESS_BATCH_SIZE = 224, 64
EPOCHS, NUM_WORKERS = 50, 4
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

def training_device() -> torch.device:
    if not torch.cuda.is_available():
        return torch.device('cpu')
    major, _ = torch.cuda.get_device_capability(0)
    if major < 7:
        print('Kaggle assigned an unsupported GPU; using CPU.')
        return torch.device('cpu')
    return torch.device('cuda')

DEVICE = training_device()
GPU_IDS = list(range(torch.cuda.device_count())) if DEVICE.type == 'cuda' else []
print({'recognition': RUN_RECOGNITION_TRAINING, 'detection': RUN_DETECTION_TRAINING, 'liveness': RUN_LIVENESS_TRAINING})
print({'device': str(DEVICE), 'gpu_count': len(GPU_IDS), 'gpu_names': [torch.cuda.get_device_name(index) for index in GPU_IDS]})


In [ ]:
# 6. Train ArcFace recognition. The saved embedder is used later for enrollment and matching.
class RecognitionDataset(Dataset):
    def __init__(self, rows, label_map, transform):
        self.rows, self.label_map, self.transform = rows, label_map, transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row['path']) as image:
            return self.transform(image.convert('RGB')), self.label_map[row['identity']]

class ArcMargin(nn.Module):
    def __init__(self, features, classes, scale=32.0, margin=0.35):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, features))
        nn.init.xavier_uniform_(self.weight)
        self.scale, self.margin = scale, margin
    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1 + 1e-7, 1 - 1e-7)
        one_hot = F.one_hot(labels, cosine.size(1)).float()
        return (cosine * (1 - one_hot) + torch.cos(torch.acos(cosine) + self.margin) * one_hot) * self.scale

class FaceEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Linear(features, 512)
    def forward(self, images):
        return F.normalize(self.projection(self.backbone(images)))

if RUN_RECOGNITION_TRAINING:
    all_rows = webface_rows
    identity_counts = Counter(row['identity'] for row in all_rows)
    train_ids = {row['identity'] for row in all_rows if row['split'] == 'train'}
    val_ids = {row['identity'] for row in all_rows if row['split'] == 'val'}
    eligible_ids = sorted(identity for identity, count in identity_counts.items() if count >= MIN_IMAGES_PER_IDENTITY and identity in train_ids and identity in val_ids)
    label_map = {identity: index for index, identity in enumerate(eligible_ids)}
    train_rows = [row for row in all_rows if row['split'] == 'train' and row['identity'] in label_map]
    val_rows = [row for row in all_rows if row['split'] == 'val' and row['identity'] in label_map]
    assert train_rows and val_rows, 'Need identities with at least five images in both train and val splits.'
    print({'eligible_identities': len(label_map), 'train_images': len(train_rows), 'val_images': len(val_rows)})
    train_tf = transforms.Compose([transforms.RandomResizedCrop(RECOGNITION_IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)), transforms.RandomHorizontalFlip(), transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1), transforms.RandomAffine(degrees=8, translate=(0.04, 0.04)), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3), transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), value='random')])
    eval_tf = transforms.Compose([transforms.Resize((RECOGNITION_IMAGE_SIZE, RECOGNITION_IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
    loader_args = dict(batch_size=RECOGNITION_BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', persistent_workers=True)
    train_loader = DataLoader(RecognitionDataset(train_rows, label_map, train_tf), shuffle=True, **loader_args)
    val_loader = DataLoader(RecognitionDataset(val_rows, label_map, eval_tf), shuffle=False, **loader_args)
    embedder, head = FaceEmbedder().to(DEVICE), ArcMargin(512, len(label_map)).to(DEVICE)
    if len(GPU_IDS) > 1:
        embedder, head = nn.DataParallel(embedder, device_ids=GPU_IDS), nn.DataParallel(head, device_ids=GPU_IDS)
    optimizer = torch.optim.AdamW(list(embedder.parameters()) + list(head.parameters()), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler, best_accuracy = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda'), 0.0
    for epoch in range(1, EPOCHS + 1):
        embedder.train(); head.train()
        for images, labels in tqdm(train_loader, desc=f'Recognition {epoch}/{EPOCHS}'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                loss = F.cross_entropy(head(embedder(images), labels), labels, label_smoothing=0.05)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        embedder.eval(); head.eval(); correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                logits = head(embedder(images.to(DEVICE)), labels.to(DEVICE))
                correct += (logits.argmax(1) == labels.to(DEVICE)).sum().item(); total += len(labels)
        scheduler.step()
        accuracy = correct / max(total, 1)
        print({'epoch': epoch, 'val_accuracy': round(accuracy, 4)})
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            embedder_state = embedder.module.state_dict() if isinstance(embedder, nn.DataParallel) else embedder.state_dict()
            torch.save({'model': embedder_state, 'labels': label_map, 'image_size': RECOGNITION_IMAGE_SIZE}, CHECKPOINT_DIR / 'recognition_arcface_best.pt')


In [ ]:
# 7. Train face detector. Ultralytics reads WIDER images/labels without mixing them into recognition.
if RUN_DETECTION_TRAINING:
    import subprocess
    import yaml
    subprocess.run(['pip', '-q', 'install', 'ultralytics'], check=True)
    from ultralytics import YOLO
    split_files = {}
    for split in ('train', 'val', 'test'):
        split_file = WORK_DIR / f'wider_{split}.txt'
        split_file.write_text('\n'.join(row['image_path'] for row in detection_rows if row['split'] == split), encoding='utf-8')
        split_files[split] = str(split_file)
    dataset_yaml = WORK_DIR / 'wider_face.yaml'
    dataset_yaml.write_text(yaml.safe_dump({'path': '/', 'train': split_files['train'], 'val': split_files['val'], 'test': split_files['test'], 'names': {0: 'face'}}), encoding='utf-8')
    detector = YOLO('yolo11n.pt')
    detector_device = GPU_IDS if len(GPU_IDS) > 1 else GPU_IDS[0] if GPU_IDS else 'cpu'
    detector.train(data=str(dataset_yaml), imgsz=DETECTION_IMAGE_SIZE, batch=DETECTION_BATCH_SIZE * max(len(GPU_IDS), 1), epochs=EPOCHS, workers=NUM_WORKERS, device=detector_device, project=str(CHECKPOINT_DIR), name='face_detector', exist_ok=True)


In [ ]:
# 8. Train binary liveness classifier from real images and spoof frames.
if RUN_LIVENESS_TRAINING:
    class LivenessDataset(Dataset):
        def __init__(self, rows, transform): self.rows, self.transform = rows, transform
        def __len__(self): return len(self.rows)
        def __getitem__(self, index):
            row = self.rows[index]
            with Image.open(row['path']) as image:
                return self.transform(image.convert('RGB')), float(row['label'])
    train_rows = [row for row in liveness_rows if row['split'] == 'train']
    val_rows = [row for row in liveness_rows if row['split'] == 'val']
    transform = transforms.Compose([transforms.Resize((LIVENESS_IMAGE_SIZE, LIVENESS_IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
    args = dict(batch_size=LIVENESS_BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', persistent_workers=True)
    train_loader = DataLoader(LivenessDataset(train_rows, transform), shuffle=True, **args)
    val_loader = DataLoader(LivenessDataset(val_rows, transform), shuffle=False, **args)
    liveness_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    liveness_model.fc = nn.Linear(liveness_model.fc.in_features, 1)
    liveness_model = liveness_model.to(DEVICE)
    if len(GPU_IDS) > 1:
        liveness_model = nn.DataParallel(liveness_model, device_ids=GPU_IDS)
    optimizer, criterion = torch.optim.AdamW(liveness_model.parameters(), lr=3e-4), nn.BCEWithLogitsLoss()
    best_accuracy = 0.0
    for epoch in range(1, EPOCHS + 1):
        liveness_model.train()
        for images, labels in tqdm(train_loader, desc=f'Liveness {epoch}/{EPOCHS}'):
            logits = liveness_model(images.to(DEVICE)).squeeze(1)
            loss = criterion(logits, labels.to(DEVICE))
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        liveness_model.eval(); correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                predictions = (liveness_model(images.to(DEVICE)).squeeze(1).sigmoid() >= 0.5).long()
                correct += (predictions == labels.to(DEVICE).long()).sum().item(); total += len(labels)
        accuracy = correct / max(total, 1)
        print({'epoch': epoch, 'val_accuracy': round(accuracy, 4)})
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            liveness_state = liveness_model.module.state_dict() if isinstance(liveness_model, nn.DataParallel) else liveness_model.state_dict()
            torch.save({'model': liveness_state, 'image_size': LIVENESS_IMAGE_SIZE}, CHECKPOINT_DIR / 'liveness_best.pt')
